# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliakhtar1010/search-ranking-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: The raw warehouse table is at one report date × pseudonymized client × pseudonymized content item. For modeling, I aggregate March 2026 into one row per client-content pair.

Feature window: March 1–31, 2026.

Outcome window: April 1–30, 2026.

Sealed test period: June 2026 remains untouched because it is the final month of the warehouse panel.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
from google.colab import userdata

# Read the private Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Start DuckDB
con = duckdb.connect()

# Let DuckDB authenticate with Hugging Face
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Hugging Face authentication ready.")

Hugging Face authentication ready.


In [2]:
rel = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("Warehouse connection ready.")

Warehouse connection ready.


In [3]:
print("Development month: 2026-03")
print("Table: fact_content_daily_performance")

Development month: 2026-03
Table: fact_content_daily_performance


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Features

march_impressions — total GSC impressions during March. Available when: computed entirely from March observations before the April outcome window.
march_clicks — total GSC clicks during March. Available when: computed entirely from March observations before the April outcome window.
march_ctr — March clicks divided by March impressions. Available when: both inputs are known from March data.
march_avg_position — impression-weighted average search position during March. Available when: calculated only from March GSC observations.
march_active_days — number of March days with available GSC data. Available when: March availability is known before predicting the April outcome.

All five features are knowable at the decision moment because they use only March 2026 observations.

Label / proxy

declined_next_month — binary proxy equal to 1 when April GSC impressions are lower than March GSC impressions, otherwise 0.

This is a proxy for future performance decline, not a true “needs refresh” label.

Context

client_hash_id
content_hash_id

These are used for grouping and joining only, never as model features.

Excluded

april_impressions and impression_change_pct are excluded from the honest model because they use outcome-window information and would leak the answer.
June 2026 is excluded because it is the sealed final test month.
GA4 fields are excluded from this first feature frame because March GA4 availability is much lower than GSC availability.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
            AS unique_grain_rows,
        COUNT(*) -
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
            AS duplicate_rows
    FROM read_parquet('{march_path}')
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows,duplicate_rows
0,9841378,9841378,0


In [6]:
slice_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM read_parquet('{march_path}')
""").df()

slice_check

,row_count,first_date,last_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [7]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows

    FROM read_parquet('{march_path}')
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score

april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/*.parquet"
)

feature_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END AS march_ctr,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS march_avg_position,

        COUNT(DISTINCT report_date) AS march_active_days

    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE

    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions

    FROM read_parquet('{april_path}')
    WHERE gsc_data_available IS TRUE

    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.march_impressions,
    m.march_clicks,
    m.march_ctr,
    m.march_avg_position,
    m.march_active_days,

    a.april_impressions,

    CASE
        WHEN a.april_impressions < m.march_impressions THEN 1
        ELSE 0
    END AS declined_next_month,

    CASE
        WHEN m.march_impressions > 0
        THEN (a.april_impressions - m.march_impressions) * 1.0
             / m.march_impressions
        ELSE NULL
    END AS impression_change_pct

FROM march m
INNER JOIN april a
    USING (client_hash_id, content_hash_id)

WHERE m.march_impressions > 0
""").df()

print("Feature frame shape:", feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (158549, 10)


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr,march_avg_position,march_active_days,april_impressions,declined_next_month,impression_change_pct
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,345.0,1.0,0.002899,23.492754,31,187.0,1,-0.457971
1,client_62f4a7e64f5e0096,content_13a8105125458098,37.0,1.0,0.027027,3.756757,17,23.0,1,-0.378378
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,15.0,0.0,0.000000,5.333333,8,17.0,0,0.133333
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,854.0,1.0,0.001171,8.152225,31,202.0,1,-0.763466
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,205.0,0.0,0.000000,34.102439,29,93.0,1,-0.546341


In [9]:
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_active_days",
]

model_df = feature_frame.dropna(
    subset=feature_cols + ["declined_next_month"]
).copy()

X = model_df[feature_cols]
y = model_df["declined_next_month"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_accuracy = accuracy_score(y_test, honest_pred)
honest_precision = precision_score(
    y_test,
    honest_pred,
    zero_division=0
)

print(f"Honest accuracy:  {honest_accuracy:.3f}")
print(f"Honest precision: {honest_precision:.3f}")

Honest accuracy:  0.657
Honest precision: 0.651


In [10]:
leaky_feature_cols = feature_cols + ["impression_change_pct"]

leaky_df = feature_frame.dropna(
    subset=leaky_feature_cols + ["declined_next_month"]
).copy()

X_leaky = leaky_df[leaky_feature_cols]
y_leaky = leaky_df["declined_next_month"]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.20,
    random_state=42,
    stratify=y_leaky
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(X_train_l, y_train_l)

leaky_pred = leaky_model.predict(X_test_l)

leaky_accuracy = accuracy_score(y_test_l, leaky_pred)
leaky_precision = precision_score(
    y_test_l,
    leaky_pred,
    zero_division=0
)

print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Leaky accuracy:  {leaky_accuracy:.3f}")

print(f"\nHonest precision: {honest_precision:.3f}")
print(f"Leaky precision:  {leaky_precision:.3f}")

Honest accuracy: 0.657
Leaky accuracy:  1.000

Honest precision: 0.651
Leaky precision:  1.000


Leakage experiment

I intentionally added impression_change_pct, which uses April outcome-window impressions. Because the proxy label is also based on whether April impressions fell below March impressions, this feature directly exposes information used to determine the label.

The leaked model therefore produces an artificially strong score. I removed impression_change_pct from the final feature set and retained the honest five-feature result.

In [11]:
final_features = feature_cols.copy()

print("Final honest feature set:")
for feature in final_features:
    print("-", feature)

Final honest feature set:
- march_impressions
- march_clicks
- march_ctr
- march_avg_position
- march_active_days


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits

The warehouse has unbalanced history, so different clients have different amounts of usable search and analytics data. March also has substantially more GSC availability than GA4 availability, which is why this first feature frame uses GSC only.

The declined_next_month target is a proxy based on month-to-month impressions. A decline in impressions does not prove that a page needs refreshing, and the data cannot prove that refreshing content would cause future traffic or ranking improvements.

The analysis therefore supports prioritization and human review rather than automatic refresh decisions or causal claims.

In [12]:
april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/*.parquet"
)

april_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{april_path}')
""").df()

april_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,first_date,last_date
0,10424730,3901060,2026-04-01,2026-04-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.